# 8장 — 갈림길 지도: 탐색과 계획

웹 챕터와 **같은 트리, 같은 수치**를 파이썬으로 계산합니다. 탐욕·빔 서치·전수 탐색이 찾는
총점과 방문 노드 수가 웹 실험실(SearchLab)과 정확히 일치해요(assert로 고정).

> 🤖 아래 셀을 실행하면 오른쪽 아래에 AI 튜터 버튼이 뜹니다 (로컬 Ollama, 없어도 학습엔 지장 없음).

In [ ]:
from tutor import chat
chat('ch8')

## 1. 갈림길 지도(탐색 트리): 3단계 선택 · 분기 3

각 경로의 총점 = 루트에서 잎까지 노드값의 합. **함정 내장**: 첫 갈림의 최고점 A(10)는
나쁜 총점으로 이어지고, 진짜 보물은 첫 갈림에서 가장 낮아 보인 C(6) 아래 숨어 있습니다.

In [ ]:
# 레벨1 노드값 (가지 A·B·C)
L1 = [10, 8, 6]
# 레벨2 노드값 (가지별 3자식)
L2 = [[3, 2, 1], [4, 3, 2], [3, 5, 1]]
# 레벨3 리프값 (가지·중간노드별 3자식)
L3 = [
    [[2, 1, 1], [1, 1, 6], [1, 1, 1]],  # A
    [[3, 15, 2], [2, 2, 1], [1, 1, 1]],  # B
    [[1, 1, 1], [2, 1, 1], [30, 1, 1]],  # C
]
NAME = ['A', 'B', 'C']

# 모든 경로 열거 (3^3 = 27)
paths = []
for a in range(3):
    for b in range(3):
        for c in range(3):
            total = L1[a] + L2[a][b] + L3[a][b][c]
            paths.append((f'{NAME[a]}{b+1}{c+1}', total, a, b, c))
print('전체 경로 수:', len(paths), '(= 3^3)')
assert len(paths) == 27

## 2. 세 전략: 탐욕 · 빔 서치 · 전수 탐색

- **탐욕**: 매 갈림에서 노드값 최고만 따라감 (k=1 빔과 같음)
- **빔 서치 k**: 부분합 기준 상위 k개 부분경로만 유지하며 전진
- **전수**: 모든 경로 평가

In [ ]:
def greedy():
    visited = 0
    a = max(range(3), key=lambda i: L1[i]); visited += 3
    b = max(range(3), key=lambda i: L2[a][i]); visited += 3
    c = max(range(3), key=lambda i: L3[a][b][i]); visited += 3
    return f'{NAME[a]}{b+1}{c+1}', L1[a] + L2[a][b] + L3[a][b][c], visited

def beam(k):
    visited = 0
    cand = [([i], L1[i]) for i in range(3)]; visited += 3
    cand = sorted(cand, key=lambda x: -x[1])[:k]
    nxt = []
    for path, s in cand:
        a = path[0]
        for i in range(3): nxt.append((path + [i], s + L2[a][i])); visited += 1
    cand = sorted(nxt, key=lambda x: -x[1])[:k]
    nxt = []
    for path, s in cand:
        a, b = path
        for i in range(3): nxt.append((path + [i], s + L3[a][b][i])); visited += 1
    win = sorted(nxt, key=lambda x: -x[1])[0]
    a, b, c = win[0]
    return f'{NAME[a]}{b+1}{c+1}', win[1], visited

def exhaustive():
    best = max(paths, key=lambda p: p[1])
    visited = 3 + 9 + 27
    return best[0], best[1], visited

for label, fn in [('탐욕', greedy), ('빔 k=2', lambda: beam(2)), ('빔 k=3', lambda: beam(3)), ('전수', exhaustive)]:
    p, tot, vis = fn()
    print(f'{label:6s} | 경로 {p} | 총점 {tot:2d} | 방문 {vis}')

# 웹 실험실과 비트 단위 일치 검증
assert greedy()[1:] == (15, 9), greedy()
assert beam(2)[1:] == (18, 15), beam(2)
assert beam(3)[1:] == (27, 21), beam(3)
assert exhaustive()[1:] == (37, 39), exhaustive()
# 총점: 탐욕 < 빔2 < 빔3 < 전수 / 방문: 같은 순서로 증가
assert 15 < 18 < 27 < 37
assert 9 < 15 < 21 < 39
print('\n\u2705 웹 실험실과 전 전략 일치. 넓게 볼수록 총점\u2191, 방문 노드\u2191')

## 3. 지수 폭발: 왜 전수 탐색은 대개 불가능한가

경로 수 = 분기^깊이 = $b^d$. 깊이가 하나 늘면 경로 수가 $b$배. 곱하기로 폭발합니다.

In [ ]:
for b, d in [(3, 3), (3, 4), (4, 5), (5, 12), (10, 10)]:
    print(f'분기 {b:2d} · 깊이 {d:2d}  \u2192  {b}^{d} = {b**d:,} 경로')

assert 3**3 == 27
assert 3**4 == 81
assert 4**5 == 1024
assert 5**12 == 244140625
print('\n\u2705 분기 5\u00b7깊이 12면 벌써 2억 4천만 경로. 전수 탐색은 사실상 불가능')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

depths = list(range(1, 13))
fig, ax = plt.subplots(figsize=(7, 4))
for b in [2, 3, 4, 5]:
    ax.plot(depths, [b**d for d in depths], marker='o', ms=3, label=f'분기 {b}')
ax.set_yscale('log')
ax.set_xlabel('깊이 d')
ax.set_ylabel('경로 수 (로그 눈금)')
ax.set_title('경로 수 = 분기^깊이 (b^d): 깊이에 따른 지수 폭발')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 오늘의 발견

- **탐욕**은 눈앞 최고만 좇다 함정에 빠진다(15점). 국소 정보로 전역 최적을 알 수 없다.
- **빔 서치**는 상위 k개 길을 들고 절충한다. k를 넓힐수록 품질↑(18→27) 비용↑(방문 15→21).
- **전수 탐색**만 진짜 최적(37)을 찾지만 방문 39개 전부. 그리고 $b^d$라 대개 불가능하다.
- 모든 탐색은 결국 **어디를 안 볼 것인가**(가지치기)의 기술: 빔 폭·되추적·미리 가보기(MCTS).

다음 장(9장 여럿이 하기)에서 갈림길을 여럿이 나눠 맡는 멀티에이전트의 수학으로 넘어갑니다.